# Designing Tool SchemasThat Actually Work

**Module 10 · Lesson 10.2**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Pydantic: One Model, Every Schema
- Descriptions Are Prompts
- The Portable Subset & the Strict-Mode Matrix
- Types, Enums & Nesting
- Granularity: Less Is More
- Side Effects & Safety

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install anthropic openai -q

# Reproducibility - the lesson uses seeded random values throughout

## The Tool That Fired for Everything

> 💡 **Info**
>
> A team ships a `get_weather` tool with the description “Weather tool.’’ In testing it looks fine. In production the model calls it for “is climate change real?,’’ “what was the temperature in Delhi in 1990?,’’ and “what’s the air quality today?’’ — none of which the tool can answer. Nothing is broken in the code; the *schema* is the bug. The model read two words and reasonably decided anything weather-adjacent belongs here. Change the description to “Get the CURRENT weather for a city. Returns temperature, humidity, and conditions. Do NOT use for forecasts, historical data, air quality, or climate questions,’’ and the over-triggering stops. That single edit — a better **schema** — is what this lesson is about: the schema is a prompt, and designing it is engineering.

### 🎯 What you will be able to do after this lesson

- **Build** type-safe tool schemas with Pydantic — one `BaseModel` giving validation, docs, and a JSON Schema you adapt to OpenAI, Anthropic, Gemini, and MCP.

- **Compare** the provider strict-mode subsets (OpenAI all-required, Gemini no-`default`/`oneOf`, Claude genuine optionals + `input_examples`, MCP full 2020-12) and design to the portable intersection — and pick types, enums, and nesting that route reliably.

- **Operate** description engineering, tool granularity (less-is-more + dynamic tool search), and side-effect safety (idempotency keys, MCP annotations, risk tiers).

- **Evaluate** schema changes with semver + golden test suites (ToolCorrectnessMetric in CI), and design Indic-correct schemas (Unicode NFC, grapheme-aware length, PAN/Aadhaar validation).

> 📦 **Info**
>
> ✅ Before you startThis builds directly on **Lesson 10.1** (the tool-use loop and the four schema levers: description, enum, required, verb_noun name). Here we go deep on each. You should be comfortable with JSON and the idea that the model only *requests* a call — your code runs it. MCP itself (what it is, and building a server) is coming up in Lessons 10.4 and 10.5.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 📝 **Analogy**
>
> **A tool schema is a job description the model reads to decide whether to apply.** “Do stuff with data’’ attracts every random query; “Senior Python dev, 3+ yrs Django, must know PostgreSQL’’ attracts exactly the right one. The **name + description** are the posting (WHEN to apply); the **parameter schema** is the required-qualifications list (WHAT to bring). **Where it breaks down:** a great posting also says who should NOT apply — that is the negative example (“Do NOT use for forecasts’’) that stops the wrong applicants, and the exact types/enums that reject a malformed application before it wastes anyone’s time.

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“A vague description is fine — the model is smart.”** It is smart, which is exactly why “Weather tool’’ fires for climate change. The description is a prompt; write it like one.
> - **“One schema works on every provider.”** Each strict mode is a different SUBSET of JSON Schema. Design to the portable intersection, then extend per provider.

> 💡 **Info**
>
> ⚠️ Anti-patternAdding an “optional’’ field to a shipped schema and assuming it is safe. Under OpenAI strict mode (`additionalProperties:false` + every property in `required`), adding **any** field is a BREAKING change — version it, do not sneak it in.

---

## 🎯 Concept 1: Pydantic: One Model, Every Schema

### Pydantic: One Model, Every Schema

Define inputs once with types, enums, and constraints. Get validation, docs, and a JSON Schema for every provider.

Do not hand-write JSON Schema. Define your tool’s inputs as a Pydantic `BaseModel` and you get three things from one source: **runtime validation** (bad args are rejected before execution), **self-documentation** (types + `Field(description=...)`), and a **JSON Schema** (`model_json_schema()`, dialect 2020-12) that you adapt into each provider’s tool format. `Literal[...]` becomes an enum; `ge/le/min_length` become constraints; a field with a default becomes optional.

> 🏭 **Analogy**
>
> Pydantic is the **architect’s master drawing**. You draw the building once; from it you print the builder’s plan (validation), the brochure (docs), and the permit filing (JSON Schema). You never redraw — you re-export. Each provider is a different government office that wants the same building on its own form.

You mark `max_price_inr` as `Optional` with a default. Pydantic’s schema lists it as NOT required. What must the OpenAI-strict adapter do?

**📝 `01_pydantic.py`** - *runnable*

In [ ]:
# ONE PYDANTIC MODEL -> validation + docs + JSON Schema for EVERY provider.
from pydantic import BaseModel, Field
from typing import Optional, Literal
import json

class SearchCourses(BaseModel):
    """Search the Netsetos course catalog by topic and price."""
    topic: Literal["genai", "python", "cloud", "data"] = Field(description="Course topic to search")
    max_price_inr: Optional[int] = Field(default=None, ge=0, le=100000,
                                         description="Max price in INR; no cap if omitted")
    sort_by: Literal["price", "rating", "duration"] = Field(default="rating", description="Sort order")

core = SearchCourses.model_json_schema()          # JSON Schema 2020-12, generated for free
print("properties:", list(core["properties"].keys()))
print("author-required:", core.get("required"))   # only fields with NO default

def to_openai(name, desc, schema):                 # STRICT subset: all-required + additionalProperties:false
    s = json.loads(json.dumps(schema)); s["additionalProperties"] = False
    s["required"] = list(s.get("properties", {}).keys())    # strict: EVERY property required
    return {"type": "function", "function": {"name": name, "description": desc, "strict": True, "parameters": s}}
def to_anthropic(name, desc, schema): return {"name": name, "description": desc, "input_schema": schema}
def to_mcp(name, desc, schema):       return {"name": name, "description": desc, "inputSchema": schema}  # full 2020-12
# gemini (google-genai): you pass the PYTHON FUNCTION; the SDK builds the declaration from hints+docstring

oa = to_openai("search_courses", "Search the catalog", core)
print("OpenAI  parameters.required (strict):", oa["function"]["parameters"]["required"])
print("Anthropic key:", list(to_anthropic("x", "", core).keys())[-1],
      "| MCP key:", list(to_mcp("x", "", core).keys())[-1])

print("validate GOOD:", SearchCourses(topic="genai", max_price_inr=15000).model_dump())
try:
    SearchCourses(topic="genai", sort_by="popularity")     # not in the Literal
except Exception as e:
    print("validate BAD:", str(e).splitlines()[0])

# Output:
# properties: ['topic', 'max_price_inr', 'sort_by']
# author-required: ['topic']
# OpenAI  parameters.required (strict): ['topic', 'max_price_inr', 'sort_by']
# Anthropic key: input_schema | MCP key: inputSchema
# validate GOOD: {'topic': 'genai', 'max_price_inr': 15000, 'sort_by': 'rating'}
# validate BAD: 1 validation error for SearchCourses

- Pydantic’s `required` lists only `topic` (the field with no default); `max_price_inr` and `sort_by` are optional there.
- The OpenAI-strict adapter rewrites `required` to ALL three properties and sets `additionalProperties:false` — the strict contract.
- Anthropic wants the core under `input_schema`; MCP under `inputSchema`; Gemini takes the Python function itself.
- Validation is real: `sort_by="popularity"` raised because it is not in the `Literal` — the same rule the model must obey.

#### 💡 What just happened

⚡ What just happened?One `BaseModel` is your single source of truth: it validates arguments at runtime, documents the tool, and emits the JSON Schema every provider consumes. You write the model once and *adapt* the core per provider — you never maintain three hand-written schemas that drift out of sync. The tradeoff is a Python dependency and a little indirection; when to use it: the moment you target two or more providers, the single-source model pays for itself, whereas a one-off single-provider tool can get by with a hand-written dict.

- Toggle fields and constraints on the Pydantic model
- Watch the OpenAI / Anthropic / Gemini / MCP schemas update

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Descriptions Are Prompts

### Descriptions Are Prompts

The model reads the description to decide WHEN to call. Capability + WHEN + returns + negative examples.

The single highest-ROI lever in tool design is the **description**, because the model reads it exactly like a prompt to decide relevance. A good description has four parts: (1) a **capability** line (“Get the current weather for a city’’), (2) **WHEN** to use it, (3) **WHAT** it returns, and (4) **negative examples** (“Do NOT use for forecasts, history, or air quality’’). Negatives are the part everyone skips and the part that stops over-triggering. Anthropic reports reaching state-of-the-art on SWE-bench in part through “precise refinements to tool descriptions’’ alone.

> 🔢 **Analogy**
>
> A description is a **road sign**. “Services’’ sends everyone off the highway looking for anything. “Fuel + EV charging, next exit; no diesel’’ sends exactly the right drivers and turns the rest away. The negative clause (“no diesel’’) is what keeps the wrong cars on the road.

Two tools, `search` and `book`, both described vaguely (“handles courses’’). The user says “Book me the cloud course.’’ What is the likely routing?

**📝 `02_descriptions.py`** - *runnable*

In [ ]:
# DESCRIPTIONS ARE PROMPTS: a tiny keyword router shows vague vs precise descriptions.
def route(query, tools):
    """Pick the best tool by keyword overlap; a NEGATIVE keyword vetoes a tool."""
    q = query.lower(); best, best_score = "answer-directly", 0
    for name, keywords, negatives in tools:
        if any(n in q for n in negatives):        # "Do NOT use for..." blocks a wrong match
            continue
        score = sum(1 for k in keywords if k in q)
        if score > best_score:
            best, best_score = name, score
    return best

VAGUE = [("search", ["course", "weather", "data", "info"], []),      # overlapping, no WHEN, no negatives
         ("book",   ["course", "weather", "data"], [])]
PRECISE = [
    ("search_courses", ["course", "price", "cost", "catalog", "which", "list"], ["book", "enroll", "forecast"]),
    ("book_course",    ["book", "enroll", "buy", "purchase"], ["search", "which", "list", "cost"]),
    ("get_weather",    ["weather", "temperature"], ["forecast", "history", "climate", "air quality"]),
]
qs = ["Which GenAI courses do you have and their price?", "Book me the cloud course",
      "What is the weather in Pune?", "What is the air quality in Delhi?"]
print("VAGUE descriptions:")
for q in qs: print(f"  {q[:38]:38s} -> {route(q, VAGUE)}")
print("PRECISE descriptions (capability + WHEN + negatives):")
for q in qs: print(f"  {q[:38]:38s} -> {route(q, PRECISE)}")

# Output:
# VAGUE descriptions:
#   Which GenAI courses do you have and th -> search
#   Book me the cloud course               -> search    <- WRONG (should book)
#   What is the weather in Pune?           -> search    <- WRONG (should be weather)
#   What is the air quality in Delhi?      -> answer-directly
# PRECISE descriptions (capability + WHEN + negatives):
#   Which GenAI courses do you have and th -> search_courses
#   Book me the cloud course               -> book_course
#   What is the weather in Pune?           -> get_weather
#   What is the air quality in Delhi?      -> answer-directly   <- negatives blocked get_weather

- With VAGUE descriptions, “Book me the cloud course’’ mis-routes to `search` and “weather’’ also grabs `search` — overlapping keywords, no WHEN.
- With PRECISE descriptions (capability + WHEN + negatives), every query routes to the right tool.
- The negative examples matter: “air quality’’ is BLOCKED from `get_weather` and correctly answered directly.
- The router is a keyword stand-in for the LLM — the point is the SHAPE of the fix, which the real model (next block) confirms.

**📝 `02b_real_routing.py current`** - *API*

In [ ]:
# THE REAL ROUTING TEST (illustrative; needs GOOGLE_API_KEY). CURRENT google-genai SDK.
# Minimal example only - production wraps each call in try/except with a timeout=.
from google import genai
from google.genai import types
client = genai.Client()                                    # GOOGLE_API_KEY

def search_courses(topic: str, max_price_inr: int = 0) -> dict:
    """Search the Netsetos catalog by topic and price. Use when the user asks WHICH
    courses exist, their price, or details. Do NOT use to enroll or book a course."""
    return {"results": []}

chat = client.chats.create(model="gemini-2.5-flash",
    config=types.GenerateContentConfig(tools=[search_courses]))   # automatic function calling is ON
print(chat.send_message("Which GenAI courses do you have?").text)  # the SDK routes + runs the tool

# Output: (illustrative - runs with GOOGLE_API_KEY set)

#### 💡 What just happened

⚡ What just happened?Descriptions are prompts. The four elements — capability, WHEN, returns, and (the one everyone forgets) NEGATIVE examples — fix the large majority of routing errors before you touch the model, the prompt, or the tool count. Write the description like you are briefing a new hire on exactly when to reach for this tool and when not to.

- Toggle capability / WHEN / returns / negative examples
- See which queries route correctly and which mis-fire

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: The Portable Subset & the Strict-Mode Matrix

### The Portable Subset & the Strict-Mode Matrix

Each provider accepts a different subset of JSON Schema. Design to the intersection; extend per provider.

Here is the trap that breaks cross-provider deployments: **strict mode is a RESTRICTED subset of JSON Schema, and each provider restricts differently.** **OpenAI** strict requires every property in `required`, mandates `additionalProperties:false`, forbids `oneOf`, cannot have `anyOf` at the root, and expresses optionals as a union with null. **Gemini** uses a restricted OpenAPI subset: no `default`, no `oneOf`, unreliable `anyOf`. **Claude** is the most permissive: genuine optional fields plus a unique `input_examples` array. And **MCP** is the outlier — its `inputSchema`/`outputSchema` are FULL JSON Schema 2020-12, so composition (`oneOf`/`anyOf`/`$ref`) is actually allowed. The **portable subset** is the safe intersection: flat objects with `type`, `properties`, `required`, `enum`, `description`, `additionalProperties:false`.

> 🔌 **Analogy**
>
> It is **travel adapters, but for schema features**. Not every plug fits every socket: `oneOf` is a plug OpenAI and Gemini reject. The portable subset is the two-pin plug that fits every socket in the world; provider extensions are the fatter plugs you only bring for one country (MCP’s full-2020-12 outlet).

Your schema uses `oneOf` for a param that is a string OR an int. It validates on an MCP server. Will it deploy unchanged to OpenAI strict?

**📝 `03_portability.py`** - *runnable*

In [ ]:
# THE PORTABLE SUBSET: which JSON Schema features survive each provider's strict mode.
MATRIX = {  # OK / NO / LIMITED / REQ / weak  (from each vendor's docs, Jul 2026)
    "string enum":                  {"OpenAI": "OK",  "Claude": "OK",      "Gemini": "OK",   "MCP": "OK"},
    "additionalProperties:false":   {"OpenAI": "REQ", "Claude": "OK",      "Gemini": "weak", "MCP": "OK"},
    "optional (omit from required)":{"OpenAI": "NO",  "Claude": "OK",      "Gemini": "OK",   "MCP": "OK"},
    "oneOf":                        {"OpenAI": "NO",  "Claude": "LIMITED", "Gemini": "NO",   "MCP": "OK"},
    "anyOf (nested)":               {"OpenAI": "OK",  "Claude": "OK",      "Gemini": "weak", "MCP": "OK"},
    "default keyword":              {"OpenAI": "NO",  "Claude": "NO",      "Gemini": "NO",   "MCP": "OK"},
    "input_examples":               {"OpenAI": "NO",  "Claude": "OK",      "Gemini": "NO",   "MCP": "NO"},
}
def portable(feature):                          # safe on the strict intersection of the 3 providers
    row = MATRIX[feature]
    return all(row[p] in ("OK", "REQ") for p in ("OpenAI", "Claude", "Gemini"))

print(f"{'feature':30s}{'OpenAI':7s}{'Claude':8s}{'Gemini':7s}{'MCP':5s}portable")
for feat, row in MATRIX.items():
    print(f"{feat:30s}{row['OpenAI']:7s}{row['Claude']:8s}{row['Gemini']:7s}{row['MCP']:5s}{portable(feat)}")

# Output:
# feature                       OpenAI Claude  Gemini MCP  portable
# string enum                   OK     OK      OK     OK   True
# additionalProperties:false    REQ    OK      weak   OK   False
# optional (omit from required) NO     OK      OK     OK   False
# oneOf                         NO     LIMITED NO     OK   False
# anyOf (nested)                OK     OK      weak   OK   False
# default keyword               NO     NO      NO     OK   False
# input_examples                NO     OK      NO     NO   False
# -> portable core = flat objects with type/properties/required/enum/description. MCP (full 2020-12) is the outlier.

- Only `string enum` is portable across all three strict providers — that is how small the safe intersection is.
- `oneOf` is `NO` on OpenAI and Gemini; `default` is ignored by all three providers; only MCP accepts them.
- `input_examples` is Claude-only; `additionalProperties:false` is REQUIRED by OpenAI but weak on Gemini.
- Strategy: author the portable core, then layer provider-specific extensions behind adapters (Step 1) rather than forking the whole schema.

#### 💡 What just happened

⚡ What just happened?The strict-mode subsets are the #1 source of cross-provider tool failures. Design to the portable intersection — flat objects, `type`/`properties`/`required`/`enum`/`description`/`additionalProperties:false` — then add extensions per provider. And remember the asymmetry: MCP is MORE permissive than the API strict modes, so an MCP-valid schema is not automatically OpenAI-valid. The tradeoff of the portable subset is expressiveness: you give up `oneOf`/`default`/composition for cross-provider reliability, so use the portable core when you ship to many providers and reach for the richer per-provider (or full-2020-12 MCP) features only when you have committed to one target.

- Pick a JSON Schema feature or a provider
- See accept / reject across OpenAI / Claude / Gemini / MCP

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Types, Enums & Nesting

### Types, Enums & Nesting

String enums beat booleans; flatten deep object trees; know that nested call sequences are a separate, hard problem.

Three type-level rules move reliability. First, prefer **string enums** over booleans or integer codes: `state:"on"|"off"` instead of two booleans that can contradict each other (on=true AND off=true) or be empty. Second, **flatten** deep object nesting — keep schemas to one or two levels, because deep trees hurt the model’s ability to fill arguments correctly. Third, do not confuse that with **nested call SEQUENCES** (the output of one call feeding the next): those are genuinely hard — on IBM’s NESTFUL benchmark the best model scored about 28% full-sequence-match. Two different problems: deep object schemas (a design choice you control) and chained calls (a model limitation you plan around).

> 🧩 **Analogy**
>
> A boolean pair is a **light switch wired with two buttons** labelled ON and OFF — nothing stops someone pressing both. A string enum is a **rotary dial**: it physically can only point at one setting. Flattening nesting is putting the important dials on the dashboard, not three menus deep in a glovebox.

A tool takes `toggle_light(on: bool, off: bool)`. How many argument combinations are possible, and how many are contradictory?

**📝 `04_enums_nesting.py`** - *runnable*

In [ ]:
# TYPES, ENUMS & NESTING: string enums beat booleans; flatten deep object trees.
def boolean_states():                           # two bools -> 4 combos, 2 of them contradictory
    combos = [(on, off) for on in (True, False) for off in (True, False)]
    invalid = [(on, off) for (on, off) in combos if on == off]   # on==off is a contradiction
    return combos, invalid

def schema_depth(d, level=1):
    if not isinstance(d, dict): return level - 1
    subs = [schema_depth(v, level + 1) for v in d.values() if isinstance(v, dict)]
    return max([level] + subs)

combos, invalid = boolean_states()
print(f"toggle_light(on:bool, off:bool): {len(combos)} states, {len(invalid)} contradictory -> {invalid}")
print("toggle_light(state: 'on'|'off'|'toggle'): 3 states, 0 contradictory  (string enum wins)")

flat   = {"user_city": "Pune", "user_zip": "411001"}                 # depth 1
nested = {"user": {"address": {"city": "Pune", "zip": "411001"}}}    # depth 3
print("flat depth  =", schema_depth(flat), "(recommended 1-2)")
print("nested depth=", schema_depth(nested), "(flatten it)")
print("NESTFUL (IBM, EMNLP 2025): best model ~28% on nested CALL SEQUENCES - a different problem from deep schemas.")

# Output:
# toggle_light(on:bool, off:bool): 4 states, 2 contradictory -> [(True, True), (False, False)]
# toggle_light(state: 'on'|'off'|'toggle'): 3 states, 0 contradictory  (string enum wins)
# flat depth  = 1 (recommended 1-2)
# nested depth= 3 (flatten it)
# NESTFUL (IBM, EMNLP 2025): best model ~28% on nested CALL SEQUENCES - a different problem from deep schemas.

- Two booleans yield 4 states, 2 of them contradictory (`(True,True)`, `(False,False)`); the string enum has exactly the valid states.
- The flat schema is depth 1; the nested one is depth 3 — flatten it (use `user_city` or one cohesive `address` object).
- NESTFUL’s ~28% is about nested CALL SEQUENCES (chaining), not deep object schemas — do not conflate the two.
- Arrays: always give `items` a type, and since `minItems/maxItems` are hint-only in strict mode, state the bound in the description.

#### 💡 What just happened

⚡ What just happened?Prefer string enums (they make invalid states unrepresentable), flatten object nesting to 1-2 levels (deep trees hurt arg-filling), and put `default`/array-bounds in the DESCRIPTION because strict mode ignores those keywords. And keep the two “nesting’’ problems separate: deep object schemas versus hard-to-chain call sequences.

- Slide the nesting depth and toggle enum vs boolean
- See predicted reliability and the flat-vs-nested path view

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Granularity: Less Is More

### Granularity: Less Is More

Fewer, well-designed tools beat big catalogs. Beyond a couple dozen, load them dynamically.

The most counterintuitive finding in tool design: **fewer tools is more accurate.** Teams that audit usage routinely find a small core does almost all the work, strip ~80% of the catalog, and watch selection accuracy climb and hallucinations drop. A rough sweet spot is a couple dozen *active* tools. Past that, you do not delete capability — you **load it dynamically**. Anthropic’s [Tool Search Tool](https://www.anthropic.com/engineering/advanced-tool-use) (mark tools `defer_loading:true`) preserves roughly 85% of the context and lifted Opus 4.5 from about 79.5% to 88.1% on large-library MCP evals; OpenAI’s **tool_search** (Responses API) does the same by loading only the relevant namespaced tools per request. Name tools `verb_noun` (`get_weather`, `create_order`) and namespace by domain (`slack_search_messages`).

> 🗡️ **Analogy**
>
> It is a **kitchen mise en place**. A line cook does not keep all 200 restaurant tools on the counter — just the dozen for tonight’s service; the rest are in the drawer, fetched when the order calls for them. Dynamic tool search is the drawer: full library available, only the relevant few on the counter.

You have 50 tools and accuracy is falling. You turn on dynamic tool search that keeps ~15 active. What happens to selection accuracy?

**📝 `05_granularity.py`** - *runnable*

In [ ]:
# LESS IS MORE: a small active tool set stays accurate; dynamic search caps it.
def selection_accuracy(n_active):
    """Toy model: selection accuracy peaks near a small active set and decays as tools pile up."""
    penalty = 0.006 * max(0, n_active - 15)     # each tool past ~15 chips away at selection
    return round(max(0.35, 0.97 - penalty), 2)
def active_set(total_tools, dynamic_search, cap=15):
    return min(total_tools, cap) if dynamic_search else total_tools

for total, dynamic in [(8, False), (50, False), (50, True)]:
    a = active_set(total, dynamic)
    label = "dynamic tool-search ON" if dynamic else "all tools static"
    print(f"{total:2d} tools, {label:22s} -> active={a:2d}, selection acc ~{selection_accuracy(a)}")
print("Anthropic Tool Search (defer_loading): ~85% context saved; Opus 4.5 79.5% -> 88.1% on large-library MCP evals.")

# Output:
#  8 tools, all tools static       -> active= 8, selection acc ~0.97
# 50 tools, all tools static       -> active=50, selection acc ~0.76
# 50 tools, dynamic tool-search ON -> active=15, selection acc ~0.97
# Anthropic Tool Search (defer_loading): ~85% context saved; Opus 4.5 79.5% -> 88.1% on large-library MCP evals.

- 8 tools score ~0.97; the same model with 50 static tools falls to ~0.76 in this toy model — choice overload.
- Turn on dynamic search (active capped at ~15) and accuracy recovers to ~0.97 while the full 50 stay available.
- The real numbers back it: Anthropic Tool Search preserves ~85% of context and lifted Opus 4.5 79.5% → 88.1%.
- Naming + namespacing (`verb_noun`, `slack_*`) also helps selection — clear names are cheap accuracy.

#### 💡 What just happened

⚡ What just happened?Tool design is subtractive. Keep a small, well-named ACTIVE set; beyond a couple dozen tools, do not fork the agent — defer-load the long tail with Anthropic’s Tool Search or OpenAI’s `tool_search`. More tools is not more capability; it is more ways to pick wrong. The tradeoff of dynamic search is a discovery round-trip and a little latency, so it is not worth it for a small toolset; use it only past a couple dozen tools, where the accuracy and token savings clearly outweigh the extra hop.

- Slide the tool count; toggle dynamic tool search
- Watch the accuracy curve and the active set change

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Side Effects & Safety

### Side Effects & Safety

Idempotency keys stop double-charges; MCP annotations classify side effects; risk tiers gate human approval.

A schema is not just a shape — it declares what a tool DOES to the world. Two rules keep agents safe. First, every WRITE tool takes an **idempotency_key**: agents retry (timeouts, self-correction), and without a key a retry double-charges or double-sends. The server records processed keys and returns the cached result on a duplicate. Second, classify side effects with **MCP tool annotations** — `readOnlyHint`, `destructiveHint` (defaults to true, the conservative choice), `idempotentHint`, `openWorldHint` — and map them to **risk tiers**: A (irreversible: delete/payment/email) needs human approval, B (reversible writes) recommends it, C (read-only/idempotent) auto-approves.

Your payment tool is marked `idempotentHint: true` and takes an `idempotency_key`. The agent retries the exact same call after a network blip. Result?

**📝 `06_side_effects.py`** - *runnable*

In [ ]:
# SIDE EFFECTS: idempotency keys stop double-charges; MCP annotations set the approval tier.
processed = {}
def charge(idempotency_key, amount_inr):
    if idempotency_key in processed:                        # agent retried the SAME key
        return {"status": "duplicate ignored", "amount": processed[idempotency_key]}
    processed[idempotency_key] = amount_inr
    return {"status": "charged", "amount": amount_inr}

def risk_tier(annotations):                                 # MCP tool annotations -> human-in-the-loop tier
    if annotations.get("readOnlyHint"):            return "C (auto-approve): read-only"
    if annotations.get("destructiveHint", True):   return "A (human approval): irreversible"
    return "B (recommended approval): reversible write"

print("agent retries a payment with the SAME idempotency_key:")
print(" ", charge("pay-8821", 14999))          # first call charges
print(" ", charge("pay-8821", 14999))          # retry is ignored -> no double charge
for name, ann in [("get_weather", {"readOnlyHint": True}),
                  ("update_profile", {"readOnlyHint": False, "destructiveHint": False, "idempotentHint": True}),
                  ("delete_account", {"readOnlyHint": False, "destructiveHint": True})]:
    print(f"  {name:15s} -> {risk_tier(ann)}")

# Output:
# agent retries a payment with the SAME idempotency_key:
#   {'status': 'charged', 'amount': 14999}
#   {'status': 'duplicate ignored', 'amount': 14999}
#   get_weather     -> C (auto-approve): read-only
#   update_profile  -> B (recommended approval): reversible write
#   delete_account  -> A (human approval): irreversible

- The first `charge("pay-8821", ...)` charges; the identical retry returns `duplicate ignored` — no second charge.
- `risk_tier` reads the MCP annotations: read-only → C (auto), reversible write → B, destructive → A (human approval).
- `destructiveHint` defaults to `true` on purpose: unknown tools are treated as dangerous until proven safe.
- This is how you wire human-in-the-loop: the schema’s annotations decide which calls pause for approval.

#### 💡 What just happened

⚡ What just happened?The schema encodes danger. Give every write tool an `idempotency_key` so agent retries cannot double-act, and annotate side effects (`readOnlyHint`/`destructiveHint`/`idempotentHint`) so a risk tier can gate irreversible actions behind human approval. Safety is a schema-design decision, not an afterthought. The cost of idempotency is storing processed keys and a little bookkeeping; the alternative — trusting retries not to double-act — is simply not safe, so for any write tool the tradeoff always favors the key.

---

## 🎯 Concept 7: Versioning & Evaluation

### Versioning & Evaluation

Schemas are contracts. Semver them, know what is breaking, and gate CI on golden tests.

A tool schema is a production contract, and changing it can break agents *silently* — no error, just wrong behavior. Version schemas with **semver**: major = breaking (rename a param, change a type, make optional required), minor = additive, patch = descriptions. The trap: under OpenAI strict mode (`additionalProperties:false`), adding **any** field is breaking. Guard against silent regressions with a **golden test suite** — 20-30+ example queries with the expected tool call — scored by a metric like DeepEval’s **ToolCorrectnessMetric** (correct tools / total called) and gated in CI. A drop of even ~5% IS a regression. Version the prompt, tools, and model together as one deployable unit. (And note: the same portable schema is exactly what an MCP server exposes as its `inputSchema` — the mapping is mechanical; the deep MCP build is Lesson 10.5.)

> 📑 **Analogy**
>
> Tool schemas are **API contracts with your model as the client**. You would never rename a REST field in place and redeploy — you version it and run the test suite. Golden tests are that suite; the ToolCorrectnessMetric is the pass/fail; CI is the gate that stops a 5% regression reaching users.

You improve a tool’s DESCRIPTION (no fields change) and separately RENAME a parameter. Which is breaking under semver?

**📝 `07_versioning_eval.py`** - *runnable*

In [ ]:
# VERSIONING & EVAL: classify breaking changes; gate CI on a golden test suite.
def is_breaking(change, strict_mode):
    hard = {"rename param", "change type", "optional->required", "remove param"}
    if change in hard:       return True
    if change == "add field": return strict_mode    # strict additionalProperties:false -> adding a field breaks
    return False                                     # improve description / add enum value = safe

def tool_correctness(expected, actual):              # DeepEval-style: correct / total called
    total = len(actual) or 1
    return round(len(set(expected) & set(actual)) / total, 2)

def golden_gate(scores, threshold=0.90):
    return "PASS" if min(scores) >= threshold else "BLOCK deploy (regression)"

for ch in ["add field", "improve description", "rename param", "optional->required"]:
    print(f"{ch:22s} strict={is_breaking(ch, True)}  lenient={is_breaking(ch, False)}")
print("ToolCorrectnessMetric:", tool_correctness(["search_courses"], ["search_courses"]),
      "vs", tool_correctness(["search_courses"], ["get_weather", "search_courses"]))
print("golden gate over 3 versions:", golden_gate([0.94, 0.91, 0.86]), "(a 5% drop is a regression)")

# Output:
# add field              strict=True  lenient=False
# improve description    strict=False  lenient=False
# rename param           strict=True  lenient=True
# optional->required     strict=True  lenient=True
# ToolCorrectnessMetric: 1.0 vs 0.5
# golden gate over 3 versions: BLOCK deploy (regression) (a 5% drop is a regression)

- The classifier: `rename param` and `optional->required` are breaking anywhere; `add field` is breaking ONLY under strict mode; a description tweak is always safe.
- `ToolCorrectnessMetric` is `correct/total`: 1.0 when only the right tool is called, 0.5 when an extra wrong tool sneaks in.
- The golden gate BLOCKS the deploy because one version scored 0.86 — below the 0.90 bar; even a ~5% drop counts as a regression.
- Ship prompt + tools + model as one versioned unit so a “small’’ schema tweak cannot silently degrade routing.

#### 💡 What just happened

⚡ What just happened?Treat schemas as versioned contracts. Semver them, remember that under strict mode even adding a field is breaking, and gate every deploy on a golden test suite scored by a tool-selection metric. Silent schema drift is one of the most common ways a working agent quietly starts failing in production. The tradeoff is maintenance: golden tests and versioning cost effort up front, but the alternative is a silent regression reaching users, so the cost is worth it for any tool real users depend on.

- Pick a schema change and strict vs lenient mode
- See the breaking verdict and the golden-test pass gate

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 8: India Schema Design

### India Schema Design

Bhashini as one tool, Unicode NFC, grapheme-aware length, and Indian identifier validation.

Indic apps add correctness requirements a demo never surfaces. Wrap **Bhashini**’s 3-step pipeline (search → config → compute) behind a single tool with a language `enum` (ISO-639-1: hi, bn, ta, te, mr, gu, kn, ml, pa, or, en, …). Normalize Indic text to **Unicode NFC** before comparing or storing — the same character can arrive as one code point or as base + combining mark, and only NFC makes them equal. Watch **length**: a conjunct like क्ष is three code points but one visible grapheme, so a `maxLength` that counts code points silently allows far fewer visible characters — use grapheme-aware validation. And validate Indian identifiers with real patterns: **PAN**, **Aadhaar**, **GSTIN**, **UPI**, mobile, pincode. Support **code-mixing** (Hinglish) as a valid “mixed’’ mode — real users switch scripts mid-sentence.

Two Hindi strings look identical on screen but one is a single code point and the other is base + nukta. Are they equal in a naive string comparison?

**📝 `08_india.py`** - *runnable*

In [ ]:
# INDIA: Unicode NFC + grapheme-aware length + Indian identifier validation.
import unicodedata, re
precomposed = "क़"          # a single code point (nukta form)
decomposed  = "क़"    # base + nukta (two code points)
print("precomposed == decomposed?", precomposed == decomposed,
      "-> after NFC:", unicodedata.normalize("NFC", precomposed) == unicodedata.normalize("NFC", decomposed))

ksha = "क्ष"     # one conjunct built from 3 code points, 1 visible grapheme
print(f"conjunct code points = {len(ksha)} but 1 grapheme -> maxLength (code points) overcounts; use grapheme-aware checks")

PATTERNS = {"PAN": r"^[A-Z]{5}[0-9]{4}[A-Z]$",
            "GSTIN": r"^[0-9]{2}[A-Z]{5}[0-9]{4}[A-Z][0-9A-Z]Z[0-9A-Z]$",
            "mobile": r"^\+91[6-9][0-9]{9}$",
            "pincode": r"^[1-9][0-9]{5}$"}
samples = {"PAN": "ABCDE1234F", "GSTIN": "27ABCDE1234F1Z5", "mobile": "+919876543210", "pincode": "411001"}
for key, pat in PATTERNS.items():
    print(f"  {key:8s} {samples[key]:16s} -> {'valid' if re.match(pat, samples[key]) else 'INVALID'}")

# Output:
# precomposed == decomposed? False -> after NFC: True
# conjunct code points = 3 but 1 grapheme -> maxLength (code points) overcounts; use grapheme-aware checks
#   PAN      ABCDE1234F       -> valid
#   GSTIN    27ABCDE1234F1Z5  -> valid
#   mobile   +919876543210    -> valid
#   pincode  411001           -> valid

- The two nukta forms are NOT equal as raw code points but ARE equal after `unicodedata.normalize("NFC", ...)` — always normalize before compare/store.
- The conjunct is 3 code points but 1 grapheme, so a code-point `maxLength` overcounts — use grapheme-aware length checks for Indic text.
- The identifier regexes (PAN/GSTIN/mobile/pincode) validate the sample values — catch malformed IDs at the schema boundary, not in the database.
- Bhashini’s 3 API calls collapse into ONE tool the model sees, with the language as an enum it cannot get wrong.

#### 💡 What just happened

⚡ What just happened?Indic-correct schemas need three things a US-centric design skips: **Unicode NFC** normalization (identical-looking text is not byte-identical), **grapheme-aware length** (code points overcount conjuncts), and **Indian identifier validation** (PAN/Aadhaar/GSTIN). Wrap multi-step Indian services like Bhashini behind a single enum-typed tool so the model has one clean thing to call. When to use these rules: any Indic-facing app; the limitation of a code-point `maxLength` is exactly why grapheme-aware checks are the alternative you reach for, and NFC normalization is the cost-free habit that prevents a whole class of comparison bugs.

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture
> A tool schema is a job description the model reads (cold-open). Define it ONCE with Pydantic (Step 1); write the description like a prompt with negative examples (Step 2); design to the portable subset because each strict mode is different (Step 3); prefer string enums and flatten nesting (Step 4); keep a small active tool set and defer-load the rest (Step 5); encode side effects with idempotency keys + annotations (Step 6); version and golden-test the schema as the contract it is (Step 7); and make it Indic-correct with NFC + identifier validation (Step 8). Every one of these is the difference between a schema that demos and a schema that survives production.

> 📦 **Info**
>
> ➡️ Where this goes nextNext up, in Lesson 10.3 you orchestrate MULTIPLE tools together (chaining, parallel calls, agent loops) — where good schemas pay off the most. The protocol itself — what MCP is and how to build a server that exposes these schemas as `inputSchema` — comes in Lessons 10.4 and 10.5. The reference is the [MCP Tools specification](https://modelcontextprotocol.io/specification/draft/server/tools).

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Designing Tool SchemasThat Actually Work**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-10_2.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-10.2.html` - regenerate this notebook whenever the source HTML is updated.*
